In [5]:
import numpy as np
import pandas as pd

# ---------- CONFIG ----------
features_a_path = "../pilot1_a_b/features_ab.csv"
features_b_path = "a_b_synthetic_features_data.csv"
labels_a_path   = "../pilot1_labels.csv"
labels_b_path   = "a_b_synthetic_labels.csv"

out_features_path = "a_b_mixed_features_data.csv"
out_labels_path   = "a_b_mixed_labels.csv"

ORIG_SHAPE = (3, 4, 2, 8)  # only for A

def process_features_a(csv_path):
    df = pd.read_csv(csv_path)
    data = df.values

    processed = []

    for row in data:
        half = row.shape[0] // 2

        real = row[:half]
        imag = row[half:]

        # reshape separately
        real = real.reshape(ORIG_SHAPE)
        imag = imag.reshape(ORIG_SHAPE)

        # stack into complex dim
        X = np.stack([real, imag], axis=-1)  # (3,4,2,8,2)

        # --- your reduction ---
        X = X[:, 0:1, :, :, :]              # (3,1,2,8,2)
        X = X.mean(axis=3, keepdims=True)   # (3,1,2,1,2)

        processed.append(X.flatten())

    return np.array(processed)

# ---------- LOAD ----------
features_a = process_features_a(features_a_path)

# B is already correct — just load
features_b = pd.read_csv(features_b_path).values

labels_a = pd.read_csv(labels_a_path).values
labels_b = pd.read_csv(labels_b_path).values


# ---------- SANITY CHECK (IMPORTANT) ----------
print("A shape:", features_a.shape)
print("B shape:", features_b.shape)

if features_a.shape[1] != features_b.shape[1]:
    raise ValueError("Feature dimension mismatch after processing!")


# ---------- COMBINE ----------
features_combined = np.concatenate([features_a, features_b], axis=0)
labels_combined   = np.concatenate([labels_a, labels_b], axis=0)


# ---------- SAVE ----------
pd.DataFrame(features_combined).to_csv(out_features_path, index=False)
pd.DataFrame(labels_combined).to_csv(out_labels_path, index=False)

print("Saved combined dataset.")
print(features_a.shape[1], features_b.shape[1])
print(np.prod(ORIG_SHAPE))

A shape: (111, 12)
B shape: (500, 12)
Saved combined dataset.
12 12
192
